In [23]:
import numpy as np
import pandas as pd
from initial_main1 import main
import random
import os
from initial import initial1,initial2,initial3,initial3_1,initial3_3

# 设置 Pandas 显示选项，以便显示所有行和列
pd.set_option('display.max_rows', None)  # 显示所有行
pd.set_option('display.max_columns', None)  # 显示所有列

log_pop_size = main.pop_size # 種群大小
log_5d = 5 # 維度
# 紀錄次數 = 迭代次數次(初始解)
run_time = main.run_time # 執行次數
problem_time = main.problem1 # 問題是prob = [01,02,...,30]
# r_para = main.r2
algorithm_select = main.algorithm_select

print(algorithm_select)

['initial1', 'initial2', 'initial3', 'initial3_1', 'initial3_3']


1. **輪盤法（Roulette Wheel Selection）**：這是最經典的選擇方法之一，個體被選擇的概率與其適應度成正比。在輪盤法中，適應度高的個體更有可能被選中。

2. **錦標賽選擇（Tournament Selection）**：這種方法將個體隨機分成小組（錦標賽），每個小組中的個體相互競爭，適應度最高的個體獲勝並被選中。

3. **隨機選擇（Random Selection）**：這是一種簡單的方法，每個個體都有相等的機會被選中，不考慮其適應度。

4. **比例選擇（Proportional Selection）**：與輪盤法類似，但不是完全按適應度選擇。個體的適應度值被歸一化，並且在選擇時根據歸一化的適應度值來選擇個體。

5. **指數選擇（Exponential Selection）**：適應度高的個體獲得更高的概率被選擇，但相對於輪盤法，適應度差異更明顯。

6. **排名選擇（Rank Selection）**：個體根據其適應度排名來選擇，排名高的個體獲得更高的選擇概率。

7. **剪枝選擇（Stochastic Universal Sampling）**：與輪盤法類似，但使用均勻分佈來選擇個體，以減少隨機性。

8. **自適應選擇（Adaptive Selection）**：根據進化過程中個體的適應度動態調整選擇概率，以更好地適應搜索空間。

這些方法在不同情況下表現不同，通常需要根據具體問題和算法的性質來選擇合適的選擇方法。輪盤法是最常用的方法之一，但其他方法也在特定情況下具有優勢。

# 0. 初始解分析
分析算法生成的解（solution）和它們的適應值（fitness）是非常重要的，它可以幫助你了解算法的性能，進行改進和優化。以下是一些分析解的方法：

1. **適應度分佈分析**：繪製適應度值的直方圖或箱線圖，以了解解決方案的分佈情況。這有助於確定是否存在局部最優解或離群值。

2. **最佳解**：確定最佳解的適應度和特徵，以及它們在迭代過程中的變化。這有助於了解算法是否在不斷改進解。

3. **收斂分析**：跟踪算法的收斂情況，即適應度是否趨於穩定。你可以繪製適應度隨時間或迭代次數的變化圖表。

4. **參數敏感性分析**：改變算法的參數（如果有的話），觀察它們對解和適應值的影響。這有助於優化參數選擇。

5. **統計分析**：使用統計方法對解決方案的性能進行分析，如均值、方差、標準差等。這可以提供有關解性能的數值摘要。

6. **對比實驗**：與其他算法或不同參數設置的算法進行對比，以確定哪種方法在解決特定問題上表現最好。

7.  **可解釋性分析**：如果可能的話，解釋模型或算法生成的解，以理解為什麼某些解更優越。

## 這次資料分析的工作項目

* 動態變數跑批次檔
* 每個個體在迭代過程的狀態(解，適應值)輸出到.csv[X]
* 會有排序問題，要重新整理成可追蹤模式[X]
* 要分成幾個 df 分別儲存第 n 次迭代的資料[X]
* 印出點陣圖檢查有沒有過早收斂(檢查每次迭代適應值的標準差) => 繪製標準差收斂曲線[]
* 輸出位置更新的狀態，檢查有無重複移動[]
* 更新 cp 值公式[]
* 紀錄算法運行時間[X]
* 紀錄可行解與不可行解的數量[X]

* 現在初始解中不可行的初始解太高導致必須要靠修復函數進而導致種群多樣性不足[]
* 思考CP 值公式的優化[]

* 種群在迭代過程的收斂狀態，針對適應值
* 評估種群決策多樣性(找找看多樣性有沒有什麼統計分析)
* 整理每個個體在位置更新時移動軌跡(決策變動的過程)


In [24]:
log_df = {} # 紀錄運行過程
best_df = {} # 紀錄最佳解
# 批次讀取 CSV 文件，動態生成變量
for i in range(run_time):
    for prob in main.problem1:
        for name in algorithm_select:
            log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"] = pd.read_csv(f"output/{name}/log_runtime_{i}_problem_{prob}_algorithm_{name}.csv") # 更新紀錄
            best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"] = pd.read_csv(f"output/{name}/best_runtime_{i}_problem_{prob}_algorithm_{name}.csv") # 最佳解
            # 名稱
            log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"].rename(columns = {log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"].columns[0]:"編號"}, inplace = True) # 改變欄位名稱
            log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"].rename(columns = {log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"].columns[-1]:"適應值"}, inplace = True) # 改變欄位名稱
            best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"].rename(columns = {best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"].columns[0]:"編號"}, inplace = True) # 改變欄位名稱
            best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"].rename(columns = {best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"].columns[-1]:"適應值"}, inplace = True) # 改變欄位名稱
            # 數值轉換
            log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"] = log_df[f"log_run_{i}_prob_{prob}_algorithm_{name}"].astype(float).round(1) # 轉換
            best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"] = best_df[f"best_run_{i}_prob_{prob}_algorithm_{name}"].astype(float).round(1) # 轉換       

In [30]:
# 顯示資料
log_df["log_run_0_prob_01_algorithm_initial1"].head(20) # 顯示資料
# log_df["log_run_0"].columns[0] # 列名

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
0,0.2,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,-424.0
1,0.2,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,-451.0
2,0.2,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,-455.0
3,0.2,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,-477.0
4,0.2,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,-508.0
5,0.2,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,-529.0
6,0.2,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,-556.0
7,0.2,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,-588.0
8,0.2,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,-594.0
9,0.2,1.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,-601.0


# initial 3 輪盤法+ 貪婪法

In [31]:
# 顯示資料
log_df["log_run_0_prob_01_algorithm_initial3"].head(20) # 顯示資料

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
0,15.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1496.0
1,5.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1436.0
2,8.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1261.0
3,10.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1108.0
4,14.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1106.0
5,20.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1045.0
6,12.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1017.0
7,7.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,925.0
8,19.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,834.0
9,6.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,814.0


# initial 3_1 輪盤法+ 貪婪法

In [32]:
# 顯示資料
log_df["log_run_0_prob_01_algorithm_initial3_1"].head(20) # 顯示資料

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
0,13.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1728.0
1,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1547.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,1154.0
3,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1119.0
4,16.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1116.0
5,3.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1108.0
6,12.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1107.0
7,4.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1050.0
8,17.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,921.0
9,11.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,905.0


# 3_2 輪盤法+貪婪

In [33]:
# 顯示資料
log_df["log_run_1_prob_01_algorithm_initial3_3"].head(20) # 顯示資料

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
0,20.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1540.0
1,12.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1336.0
2,16.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1335.0
3,18.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1287.0
4,8.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,1270.0
5,2.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1232.0
6,9.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1057.0
7,17.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1040.0
8,3.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1004.0
9,11.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,994.0


In [34]:
total=log_df["log_run_10_prob_01_algorithm_initial3"].iloc[:,-1].values
print(total)
sum = np.sum(total)
sum

KeyError: 'log_run_10_prob_01_algorithm_initial3'

# 1.隨機生成初始解

In [ ]:
# 顯示資料
log_df["log_run_0_prob_01_algorithm_initial1"].head(20) # 顯示資料
# log_df["log_run_0"].columns[0] # 列名

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
0,3.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1125.0
1,15.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,909.0
2,10.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,880.0
3,13.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,843.0
4,14.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,671.0
5,8.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,602.0
6,18.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,551.0
7,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,508.0
8,16.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-15.0
9,9.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,-23.0


In [ ]:
total=log_df["log_run_7_prob_01_algorithm_initial3_3"].iloc[:,-1].values
print(total)
sum = np.sum(total)
sum

[1865. 1366. 1242. 1193. 1155. 1048. 1005. 1003.  993.  876.  861.  780.
  733.  697.  692.  619.  548.  536.  523.  452.]


18187.0

# 1.初始解分析
每個決策變數生成的步驟:
* 利用亂數產生一個界於"下界" 與 "上界" 的隨機值(r_i)
* 該隨機值(r_i)透過"轉換函數" 做映射至 0 到 1 之間的值(m_i)
* 二元化該值，如果該值 (m_i) 大於某值(r) 為 1，否則為 0

問題:
* 對於該決策變數最終產生決策與問題本身毫無關係
* 以個體的角度而言，沒有考慮限制條件(為難的點是如果全部考慮限制條件會變貪婪策略，會導致種群多樣性降低造成局部陷阱產生)
* 以種群的角度而言，不可行解太多了
目標:
* 提升種群整體的適應值
* 降低不可行解的個體數量

猜想:
* 可能可以用"輪盤法"產生初始解每一個決策變數將會受到物品的客觀條件影響
* 利用調整機率的方法使種群分布決策變數為 1 的情況合理

1. **輪盤法（Roulette Wheel Selection）**：這是最經典的選擇方法之一，個體被選擇的概率與其適應度成正比。在輪盤法中，適應度高的個體更有可能被選中。

2. **錦標賽選擇（Tournament Selection）**：這種方法將個體隨機分成小組（錦標賽），每個小組中的個體相互競爭，適應度最高的個體獲勝並被選中。

3. **隨機選擇（Random Selection）**：這是一種簡單的方法，每個個體都有相等的機會被選中，不考慮其適應度。

4. **比例選擇（Proportional Selection）**：與輪盤法類似，但不是完全按適應度選擇。個體的適應度值被歸一化，並且在選擇時根據歸一化的適應度值來選擇個體。

5. **指數選擇（Exponential Selection）**：適應度高的個體獲得更高的概率被選擇，但相對於輪盤法，適應度差異更明顯。

6. **排名選擇（Rank Selection）**：個體根據其適應度排名來選擇，排名高的個體獲得更高的選擇概率。

7. **剪枝選擇（Stochastic Universal Sampling）**：與輪盤法類似，但使用均勻分佈來選擇個體，以減少隨機性。

8. **自適應選擇（Adaptive Selection）**：根據進化過程中個體的適應度動態調整選擇概率，以更好地適應搜索空間。

這些方法在不同情況下表現不同，通常需要根據具體問題和算法的性質來選擇合適的選擇方法。輪盤法是最常用的方法之一，但其他方法也在特定情況下具有優勢。

在啟發式算法中，生成初始解（或種子解）的方法取決於問題的性質和算法的需求。以下是一些常見的生成初始解的方法：

1. **隨機生成**：這是最簡單的方法之一，通過隨機選擇問題域中的值來生成初始解。雖然簡單，但通常效果較差，適用於問題的搜索空間較小時。

2. **貪婪法（Greedy Initialization）**：根據某種啟發式規則，從問題的初始狀態開始逐步構建解。例如，對於旅行商問題（TSP），可以從一個隨機城市開始，然後選擇距離最近的未訪問城市，逐步構建巡迴路線。

3. **構造性啟發式（Constructive Heuristics）**：這些方法基於問題的結構，以一種啟發式的方式逐步構建解決方案。例如，對於背包問題，可以使用動態規劃或貪婪算法來構建解。

4. **局部搜索**：使用一種局部搜索方法（如爬山法）從一個初始解開始，嘗試改進解決方案。這通常需要一個初始解作為起點。

5. **遺傳算法的初始化**：在遺傳算法中，可以使用隨機生成一組個體作為初始種群，或者根據問題的特性生成一組個體。

6. **基於歷史信息的方法**：有些問題的初始解可以根據問題的歷史信息或已知的啟發式信息來生成。

7. **問題特定的初始化方法**：針對特定問題，可以根據問題的特性設計專門的初始化方法。例如，對於調度問題，可以根據工件的特性生成初始調度。

選擇哪種方法取決於問題的性質、算法的要求以及計算資源的可用性。通常，生成高質量的初始解對於啟發式算法的性能至關重要，因為良好的初始解可以加速算法的收斂。

In [35]:
# 批次抓初始解 第{i}次執行，第{j}個問題
initial_log = {}
for i in range(run_time):
    for j in main.problem1:
        for k in range(len(initial1.r2)):
            initial_log[f"initial1_log_run_{i}_prob_{j}_r2_{k}"] = log_df[f"log_run_{i}_prob_{j}_algorithm_initial1"].iloc[20*k:20*k+20] # initial_log[變數名稱] = 內容

In [47]:
# 顯示   
initial_log["initial1_log_run_0_prob_01_r2_3"].head(20) # 訪問動態生成的變量值(输出)
# initial_log["initial_log_run_1_prob_0"] # 訪問動態生成的變量值(输出)
# initial_log["initial_log_run_2_prob_0"] # 訪問動態生成的變量值(输出)

,編號,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,適應值
60,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1588.0
61,0.5,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,-23.0
62,0.5,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,-101.0
63,0.5,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,-135.0
64,0.5,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,-138.0
65,0.5,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,-158.0
66,0.5,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-210.0
67,0.5,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,-222.0
68,0.5,0.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,-229.0
69,0.5,0.0,0.0,1.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,-234.0


In [ ]:
# 第 i 次執行中 0 或 1 的統計數量
# log_df["log_run_0_prob_01_algorithm_initial1"]
fessible_log = {} # 紀錄空間
for i in range(run_time):
    for j in main.problem1:

            # subset = initial_log[f"initial3_log_run_{i}_prob_{j}_r2_{k}"].iloc[:, 1:-1] # 初始解的決策變數
            subset = log_df[f"log_run_{i}_prob_{j}_algorithm_initial3_3"].iloc[:, 1:-1] # 初始解的決策變數
            counts_1 = (subset.values == 1).sum() # 計算是 1 的數量
            counts_0 = (subset.values == 0).sum() # 計算是 0 的數量



            fitness = log_df[f"log_run_{i}_prob_{j}_algorithm_initial3_3"].iloc[:,-1].values
            tatal_fit = np.sum(fitness)
            # print(fitness)
            fessible = 0 # 可行
            infessible = 0 # 不可行
            for l in range(log_pop_size):
                if fitness[l] < 0 :
                    infessible += 1
                else:
                    fessible += 1
            fessible_log[f"run_{i}_prob_{j}_algorithm_initial3_3"] =  [f"log_run_{i}\t prob_{j}\t algorithm_initial3 \t 不可行\t{infessible} \t 可行 \t{fessible} \t 種群總fit \t {int(tatal_fit)}\t'1'個數 \t {counts_1}\t '0'個數 \t{counts_0}"]
            print(f"run_{i},prob_{j},algorithm_initial3_3,不可行,{infessible},可行,{fessible},種群總fit,{int(tatal_fit)},'1'個數,{counts_1},'0'個數,{counts_0}")
            

run_0,prob_01,algorithm_initial3_3,不可行,0,可行,20,種群總fit,20814,'1'個數,182,'0'個數,418
run_0,prob_02,algorithm_initial3_3,不可行,0,可行,20,種群總fit,19370,'1'個數,185,'0'個數,415
run_0,prob_03,algorithm_initial3_3,不可行,0,可行,20,種群總fit,12459,'1'個數,122,'0'個數,478
run_0,prob_04,algorithm_initial3_3,不可行,0,可行,20,種群總fit,8768,'1'個數,96,'0'個數,504
run_0,prob_05,algorithm_initial3_3,不可行,0,可行,20,種群總fit,6745,'1'個數,78,'0'個數,522
run_0,prob_06,algorithm_initial3_3,不可行,0,可行,20,種群總fit,22297,'1'個數,248,'0'個數,552
run_0,prob_07,algorithm_initial3_3,不可行,0,可行,20,種群總fit,19816,'1'個數,241,'0'個數,559
run_0,prob_08,algorithm_initial3_3,不可行,0,可行,20,種群總fit,22267,'1'個數,248,'0'個數,552
run_0,prob_09,algorithm_initial3_3,不可行,0,可行,20,種群總fit,11498,'1'個數,138,'0'個數,662
run_0,prob_10,algorithm_initial3_3,不可行,0,可行,20,種群總fit,15096,'1'個數,177,'0'個數,823
run_0,prob_11,algorithm_initial3_3,不可行,0,可行,20,種群總fit,9119,'1'個數,112,'0'個數,888
run_0,prob_12,algorithm_initial3_3,不可行,0,可行,20,種群總fit,14227,'1'個數,174,'0'個數,826
run_0,prob_13,algorithm_initial3_3,不可行,0,可行,2

In [ ]:
fessible_log[f"log_run_0_prob_01_algorithm_initial3"]

["log_run_0\t prob_01\t algorithm_initial3 \t 不可行\t0 \t 可行 \t20 \t 種群總fit \t 17224\t'1'個數 \t 187\t '0'個數 \t413"]

In [ ]:
# 第 i 次執行中 0 或 1 的統計數量

fessible_log = {} # 紀錄空間
for i in range(run_time):
    for j in main.problem1:
        for k in range(len(main.r2)):
            subset = initial_log[f"initial2_log_run_{i}_prob_{j}_r2_{k}"].iloc[:, 1:-1] # 初始解的決策變數
            counts_1 = (subset.values == 1).sum() # 計算是 1 的數量
            counts_0 = (subset.values == 0).sum() # 計算是 0 的數量



            fitness = initial_log[f"initial1_log_run_{i}_prob_{j}_r2_{k}"].iloc[:,-1].values
            tatal_fit = np.sum(fitness)
            # print(fitness)
            fessible = 0 # 可行
            infessible = 0 # 不可行
            for l in range(log_pop_size):
                if fitness[l] < 0 :
                    infessible += 1
                else:
                    fessible += 1
            fessible_log[f"initial1_log_run_{i}_prob_{j}_r2_{k}"] =  [""]
            print(f": 不可行:{infessible}\t可行:{fessible}\t種群總fit:{int(tatal_fit)}\t'1'個數:{counts_1}\t'0'個數:{counts_0}\n")
            

run_0/problem_01/infessible/r2_0.2 : 不可行:20	可行:0	種群總fit:-11753	'1'個數:496	'0'個數:104

run_0/problem_01/infessible/r2_0.3 : 不可行:20	可行:0	種群總fit:-8965	'1'個數:413	'0'個數:187

run_0/problem_01/infessible/r2_0.4 : 不可行:20	可行:0	種群總fit:-5621	'1'個數:331	'0'個數:269

run_0/problem_01/infessible/r2_0.5 : 不可行:16	可行:4	種群總fit:3159	'1'個數:285	'0'個數:315

run_0/problem_01/infessible/r2_0.6 : 不可行:13	可行:7	種群總fit:9665	'1'個數:239	'0'個數:361

run_0/problem_01/infessible/r2_0.7 : 不可行:9	可行:11	種群總fit:14571	'1'個數:194	'0'個數:406

run_0/problem_01/infessible/r2_0.8 : 不可行:2	可行:18	種群總fit:19181	'1'個數:115	'0'個數:485

run_0/problem_01/infessible/r2_None : 不可行:11	可行:9	種群總fit:7650	'1'個數:292	'0'個數:308

run_0/problem_02/infessible/r2_0.2 : 不可行:20	可行:0	種群總fit:-10071	'1'個數:475	'0'個數:125

run_0/problem_02/infessible/r2_0.3 : 不可行:20	可行:0	種群總fit:-7735	'1'個數:415	'0'個數:185

run_0/problem_02/infessible/r2_0.4 : 不可行:20	可行:0	種群總fit:-6247	'1'個數:372	'0'個數:228

run_0/problem_02/infessible/r2_0.5 : 不可行:18	可行:2	種群總fit:1182	'1'個數:299	'0'個數:301

run_0

# 相關性評估

In [ ]:
# # 初始解的相關性評估
# def similarity_analysis(population):
#     similarity_matrix = [[0] * log_pop_size for _ in range(log_pop_size)]
#     for i in range(log_pop_size):
#         for j in range(i+1,log_pop_size):
#             similarity = sum([1 for k in range(log_d) if log_pop_size[i][k] == log_pop_size[j][k]])
#             similarity_matrix[i][j] = similarity
#             similarity_matrix[j][i] = similarity
    
#     return similarity_matrix

# # 測試程式碼
# population = initial_log["initial_log_run_0_prob_0"]
# similarity_matrix = similarity_analysis(population)

# # 輸出相似性矩陣
# for row in similarity_matrix:
#     print(row)


In [ ]:
import random

# 定義種群大小和解的維度
population_size = 20
solution_dimension = 30

# 生成隨機的初始解
def generate_initial_solution():
    return [random.randint(0, 1) for _ in range(solution_dimension)]

# 生成初始種群
def generate_initial_population():
    return [generate_initial_solution() for _ in range(population_size)]

# 計算解的相似性分析
def similarity_analysis(population):
    similarity_matrix = [[0] * population_size for _ in range(population_size)]
    
    for i in range(population_size):
        for j in range(i+1, population_size):
            similarity = sum([1 for k in range(solution_dimension) if population[i][k] == population[j][k]])
            similarity_matrix[i][j] = similarity
            similarity_matrix[j][i] = similarity
    
    return similarity_matrix

# 測試程式碼
population = generate_initial_population()
print(population,"\n")
similarity_matrix = similarity_analysis(population)

# 輸出相似性矩陣
# for row in similarity_matrix:
#     print(row)


[[1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1], [0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0], [0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0], [1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0], [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1], [0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0], [1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1], [0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1], [1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 

# 3.多樣性分析
多樣性分析是在算法優化和搜索領域中常用的一種技術，它有助於了解算法生成的解決方案之間的多樣性。以下是一些用於多樣性分析的常見方法：

解的分佈圖：繪製解決方案的分佈圖，通常用於二維或三維問題。這有助於可視化解的空間分佈。

解的距離分析：計算解決方案之間的距離或相似性，可以使用歐氏距離、曼哈頓距離、餘弦相似性等。較遠的解之間具有更高的多樣性。

解的聚類：應用聚類算法（如K均值聚類）來將解決方案劃分為不同的簇，每個簇表示相似的解。這有助於識別解決方案的群集。

分析前沿：在多目標優化中，繪製前沿圖，顯示不同權衡解決方案之間的多樣性。前沿圖顯示了權衡解之間的權衡關係。

種群多樣性指標：使用多樣性指標（如群集熵、多樣性指數等）來量化解決方案的多樣性水平。這些指標可以用於監視算法的多樣性進展。

解的差異度分析：比較解決方案與參考解之間的差異。參考解可以是已知的最優解或其他基準解。

變異分析：分析算法如何通過變異操作（如交叉、變異）生成不同的解。這有助於了解算法的多樣性來源。

局部搜索策略：引入局部搜索策略，以在一定程度上保留解的多樣性，例如，在搜索的早期階段採用多樣性較大的搜索策略。

可視化：使用可視化工具可視化解決方案的多樣性，例如，繪製解決方案的散點圖或散點矩陣。

敏感性分析：分析不同參數設置對解的多樣性的影響，確定哪些參數設置更有利於多樣性。

多樣性分析有助於評估算法的搜索策略，確保算法不僅找到高質量解，還具有多樣性，以避免陷入局部陷阱。這些方法通常需要與特定問題和優化算法相結合，以更好地理解解決方案的多樣性。

In [52]:
# 漢明距離
# 計算初始解的多樣性
def calculate_diversity(pop):
    num_solutions = len(pop)
    diversity = 0
    for i in range(num_solutions):
        for j in range(i+1, num_solutions):
            diversity += hamming_distance(pop.iloc[i,:], pop.iloc[j,:])
    diversity /= (num_solutions * (num_solutions - 1) / 2) # 正規化多樣性值
    return diversity

# 計算漢明距離
def hamming_distance(solution1, solution2):     
    distance = sum(bit1 != bit2 for bit1, bit2 in zip(solution1, solution2))
    return distance

# 執行計算初始解的多樣性
for i in range(run_time):
    initial_log["initial1_log_run_0_prob_01_r2_0"] = initial_log["initial1_log_run_0_prob_01_r2_0"].astype(int)
    pop = initial_log["initial1_log_run_0_prob_01_r2_0"].iloc[:, 1:-1]
    # print(pop)
    diversity = calculate_diversity(pop)
    print(f"第{i}次執行，初始解的多樣性：{diversity}") # 漢明距離


第0次執行，初始解的多樣性：9.473684210526315
第1次執行，初始解的多樣性：9.473684210526315
第2次執行，初始解的多樣性：9.473684210526315
第3次執行，初始解的多樣性：9.473684210526315
第4次執行，初始解的多樣性：9.473684210526315
第5次執行，初始解的多樣性：9.473684210526315
第6次執行，初始解的多樣性：9.473684210526315
第7次執行，初始解的多樣性：9.473684210526315
第8次執行，初始解的多樣性：9.473684210526315
第9次執行，初始解的多樣性：9.473684210526315
